# plot_05. Paper figures

The figures for a paper on the forced response: the **main figures** put the models together, and the **supplementary figures** show each model. They are drawn from what notebook calc_02 saved, with the same colour scales as notebook plot_03, in print style: 6.5–8 pt text, every figure 183 mm (7.2 in) wide (a two-column page), panel letters, and no titles (the caption above each figure is its title). *Check the Colour Ranges* says whether the scales suit the data. Each is saved as a PDF (vector) and a 300 dpi PNG in `figures/<variable>/paper/`.

| Figure | What it shows |
|---|---|
| 1 | The distribution, after Bracegirdle et al. (2024, Fig. 1): Q05, the mean, Q95 and the width in hist-nat and historical, and the change in each (multi-model median), in winter (JJA) |
| 2, 3 | How each forcing changes the distribution: the bottom row of Fig. 1 for every forcing, in winter (JJA) and summer (DJF) |
| 4 | Where the mean and the width have changed together, every forcing and both seasons, on one 3 × 3 colour key |
| 5 | Area means south of 60°S: every model's change in the mean and in the width, every forcing and season |
| 6 | The change at each latitude (after Bracegirdle et al., 2024, Fig. 2), every model as a line, with the sea-ice edge (JJA) |
| S1 | Fig. 1 in summer (DJF) |
| S2, S3 | Fig. 2 and 3 for every model on its own (historical) |
| S4–S7 | How many models have a significant change in the mean, and in the width (DJF, JJA) |
| S8, S9 | S/N of every model (DJF, JJA) |
| S10 | The change at every 5th percentile, every model (DJF, JJA) |
| S11 | Additivity: historical against the sum of the single forcings, every model (JJA) |
| S12 | Fig. 6 in summer (DJF) |

**Putting the models together.** A multi-model panel is the median across models. A change is **robust** where at least two-thirds of the models (`ROBUST`, `config.SIGNIFICANT_THRESHOLD`) have a significant change of the same sign (the IPCC AR6 approach); dots mark where it is not. The tests are those of notebook calc_02: the hist-nat bootstrap of the mean and of the width (of the internal variability), each experiment's final 21 years against 10 random hist-nat members over the same years, with the same trials for both; single quantiles have no test. Every change is measured from hist-nat over the same years.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pandas as pd
import xarray as xr

from extant import config, response_change as rc, sea_ice, significance as sig, zonal
from extant.config import ALPHA, FORCED_EXPERIMENTS, WINDOW
from extant.logger import setup_logging
from extant.plots import antarctica, charts, forced_response as fr, response_change as rcp, zonal as zp
from plotting_modules.constants import FORCING_COLORS, SEASONS

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook; the paper's
#(t): figures are drawn in fr.PAPER's
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

The settings every notebook shares are in `extant/config.py` (`ALPHA`, `FORCED_EXPERIMENTS`, `WINDOW`) and `plotting_modules/constants.py` (`SEASONS`). Only this notebook's own choices are here.

In [ ]:
VARIABLE = config.VARIABLES['tas']

#(t): The winter and summer of the main figures
WINTER, SUMMER = 'JJA', 'DJF'

#(t): |S/N| counted as emerged, and the fraction of models that makes a change robust
SN_THRESHOLD = 2
ROBUST = config.SIGNIFICANT_THRESHOLD

#(t): The test of the mean, as in notebook plot_03 section 5: 'significant' is the hist-nat bootstrap
MEAN_TEST = 'significant'

#(t): Print style; the width of every figure, a two-column page (183 mm, which each figure's panels fill); and the
#(t): most a map figure can be tall (229 mm: its panels shrink to fit, leaving room for the caption)
STYLE = fr.PAPER
WIDTH = fr.PAGE_WIDTH
HEIGHT = fr.PAGE_HEIGHT

#(t): The room above a paper figure's maps (inches): its column labels only, as it has no title
PAPER_TITLE_HEIGHT = 0.35


In [ ]:
#(t): Where notebooks calc_02 and calc_04 saved their results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

In [ ]:
#(t): The same colour scales as notebook plot_03
scales = fr.colour_scales(VARIABLE.name, alpha=ALPHA)

In [ ]:
#(t): The legend entries of the maps: the dots where the models don't agree (main figures) or a model's change is not
#(t): significant (supplementary figures)
#(c): On three lines and two, to fit beside the colour bars of a page-wide figure
not_robust_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                               label='Not robust (fewer than\n2/3 of the models\nsignificant)')
not_significant_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                                    label=f'Not significant\n(p ≥ {ALPHA:g})')

## Open the Saved Results

Run **one** of the next two sections. Both leave `RESULTS_DIR` and `ICE_RESULTS_DIR`, the folders notebook calc_02's and notebook calc_04's results are opened from, and `PAPER_DIR`, where the figures go (this repository's figures folder).

### Sample data

In [ ]:
#(t): The sample's results (notebook calc_02's, and calc_04's sea ice), and where its figures go
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'sample'
PAPER_DIR = Path('../figures') / VARIABLE.name / 'paper_sample'
RESULTS_DIR, ICE_RESULTS_DIR, PAPER_DIR

### Full data

In [ ]:
#(t): The full data's results, and where the paper's figures go
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'full'
PAPER_DIR = Path('../figures') / VARIABLE.name / 'paper'
RESULTS_DIR, ICE_RESULTS_DIR, PAPER_DIR

### Saved results

In [ ]:
#(t): The hist-nat bootstrap of the mean change, with hist-nat's and every experiment's final-years mean
mean_bootstrap_ds = xr.open_zarr(RESULTS_DIR / 'mean_bootstrap.zarr').load()

In [ ]:
#(t): S/N
lesfmip_seasonal_sn_ds = xr.open_zarr(RESULTS_DIR / 'sn.zarr').load()

In [ ]:
#(t): hist-nat's final-years quantiles
hist_nat_final_q_da = xr.open_zarr(RESULTS_DIR / 'hist_nat_final_quantiles.zarr')[VARIABLE.name].load()

In [ ]:
#(t): Every experiment's final-years quantiles
final_q_da = xr.open_zarr(RESULTS_DIR / 'final_quantiles.zarr')[VARIABLE.name].load()

In [ ]:
#(t): The change in every 5th percentile, as an Antarctic mean
quantile_curve_da = xr.open_zarr(RESULTS_DIR / 'quantile_curve.zarr')['quantile_curve'].load()

In [ ]:
#(t): The width test: the change in Q95 − Q05, and its p-value
qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'qrange_change.zarr').load()

In [ ]:
#(t): The change in each tail, Q95 − Q50 and Q50 − Q05
tails_ds = xr.open_zarr(RESULTS_DIR / 'tails.zarr').load()

In [ ]:
#(t): Each experiment's change from its own 1850-1900 baseline
own_changes_da = xr.open_zarr(RESULTS_DIR / 'own_changes.zarr')['own_change'].load()

In [ ]:
#(t): The zonal-mean test of the mean
zonal_mean_ds = xr.open_zarr(RESULTS_DIR / 'zonal_mean_test.zarr').load()

In [ ]:
#(t): The zonal-mean test of the width
zonal_width_ds = xr.open_zarr(RESULTS_DIR / 'zonal_width_test.zarr').load()

In [ ]:
#(t): The models: every count runs up to this many
N_MODELS = mean_bootstrap_ds.sizes['model']
list(mean_bootstrap_ds['model'].values)

### Sea ice

For the ice edge in Figs 6 and S12 (notebook calc_04, section 3). Run **one** of the next two sections.

#### Without sea ice

In [ ]:
ice_edges_da = None
ice_edges_da

#### With sea ice

In [ ]:
#(t): Each member's seasonal extent and its model's continent, /<model>/<experiment> (notebook calc_04)
ice_extents_tree = xr.open_datatree(ICE_RESULTS_DIR / 'extent.zarr', engine='zarr').load()

In [ ]:
#(t): The equivalent latitude of each model's edge in the final years
ice_edges_da = sea_ice.edge_latitude(ice_extents_tree, years=WINDOW)

## Fields

As in notebook plot_03, section 5: each part of the distribution in hist-nat and in every experiment, the change in each, and which changes are significant in each model.

In [ ]:
#(t): S/N in the centre of the final WINDOW years, for the 'emerged' and 'robust' mean tests
sn_final_da = lesfmip_seasonal_sn_ds[VARIABLE.name].isel(year=config.LAST_FULL_YEAR)

In [ ]:
#(t): |S/N|: how far apart the distributions have moved, whichever way (figures S8 and S9)
abs_sn_final_da = np.abs(sn_final_da)
abs_sn_final_da

In [ ]:
#(t): Every change and its test, on (model, experiment, season, lat, lon)
change_ds = rc.change_summary(mean_bootstrap_ds['mean_change'], mean_bootstrap_ds['pvalue'], sn_final_da, qrange_change_ds,
                              tails_ds, alpha=ALPHA, sn_threshold=SN_THRESHOLD)

In [ ]:
#(t): hist-nat's distribution over the final WINDOW years: Q05, the mean and Q95, and the width of its internal
#(t): variability
hist_nat_fields_ds = xr.Dataset({
    'Q05': hist_nat_final_q_da.sel(quantile=0.05, drop=True),
    'Mean': mean_bootstrap_ds['hist_nat_mean'],
    'Q95': hist_nat_final_q_da.sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['hist_nat_qrange'],
})

In [ ]:
#(t): Every experiment's, over its final WINDOW years
experiment_fields_ds = xr.Dataset({
    'Q05': final_q_da.sel(quantile=0.05, drop=True),
    'Mean': mean_bootstrap_ds['experiment_mean'],
    'Q95': final_q_da.sel(quantile=0.95, drop=True),
    'Q95 − Q05': qrange_change_ds['experiment_qrange'],
})

In [ ]:
#(t): The change in each
change_fields_ds = experiment_fields_ds - hist_nat_fields_ds

In [ ]:
#(t): Which changes are significant in each model: the mean by MEAN_TEST, the width by the hist-nat bootstrap
change_significant_ds = xr.Dataset({'Mean': change_ds[f'mean_{MEAN_TEST}'], 'Q95 − Q05': change_ds['width_significant']})

In [ ]:
#(t): Not significant, where there is data
change_not_significant_ds = ~change_significant_ds & change_fields_ds[list(change_significant_ds)].notnull()
change_not_significant_ds

In [ ]:
#(t): The sign of each model's significant changes: +1 up, -1 down, 0 not significant
mean_sign_da = sig.significant_sign(change_fields_ds['Mean'], change_significant_ds['Mean'])
width_sign_da = sig.significant_sign(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'])

In [ ]:
#(t): Each model's joint class (notebook plot_03, section 5)
model_joint_class_da = rc.joint_class(mean_sign_da, width_sign_da)
model_joint_class_da

## Models Together

The median across models of every field, and where the models agree: the sign of a change where at least `ROBUST` of the models with data have a significant change of that sign, 0 where they do not.

In [ ]:
#(t): The multi-model median of every field: hist-nat
median_hist_nat_fields_ds = hist_nat_fields_ds.median('model')
median_hist_nat_fields_ds

In [ ]:
#(t): every experiment
median_experiment_fields_ds = experiment_fields_ds.median('model')
median_experiment_fields_ds

In [ ]:
#(t): and the change
median_change_fields_ds = change_fields_ds.median('model')
median_change_fields_ds

In [ ]:
#(t): Where the models agree on a significant change in the mean
mean_robust_sign_da = sig.robust_sign(change_fields_ds['Mean'], change_significant_ds['Mean'], threshold=ROBUST)

In [ ]:
#(t): and in the width
width_robust_sign_da = sig.robust_sign(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'],
                                       threshold=ROBUST)

In [ ]:
#(t): Not robust, where there is data: the dots
not_robust_ds = xr.Dataset({'Mean': mean_robust_sign_da == 0, 'Q95 − Q05': width_robust_sign_da == 0})
not_robust_ds

In [ ]:
#(t): The joint class of the robust changes
robust_joint_class_da = rc.joint_class(mean_robust_sign_da, width_robust_sign_da)
robust_joint_class_da

In [ ]:
#(t): A check on the maps: 1 where the models agree on a change, 0 where not (NaN without data)
robust_ds = xr.Dataset({'mean': np.abs(mean_robust_sign_da), 'width': np.abs(width_robust_sign_da)})

In [ ]:
#(t): How much of the area south of 60°S each forcing robustly changes, in each season (%)
robust_area_ds = 100 * sig.area_mean(robust_ds.sel(lat=slice(None, -60)))
robust_area_ds.to_dataframe().round(0)

## Check the Colour Ranges

How much of each field falls beyond the ends of its colour scale, where the maps show only the end colour (and the colour bar an arrow). A few percent is fine. Much more means the range in `fr.RANGES` is too narrow for these data (set it there, and every notebook follows), or a model has gone wrong (notebook plot_03's *Check the results*). Every model's field is checked, not just the median, so the supplementary figures are covered too.

In [ ]:
#(t): Each field the figures draw, with the scale it is drawn with
range_checks = {
    'hist-nat: Q05, mean, Q95': (hist_nat_fields_ds[['Q05', 'Mean', 'Q95']].to_dataarray(), scales.temperature),
    'experiments: Q05, mean, Q95': (experiment_fields_ds[['Q05', 'Mean', 'Q95']].to_dataarray(), scales.temperature),
    'Q95 - Q05': (experiment_fields_ds['Q95 − Q05'], scales.spread),
    'change in Q05, mean, Q95': (change_fields_ds[['Q05', 'Mean', 'Q95']].to_dataarray(), scales.change),
    'change in Q95 - Q05': (change_fields_ds['Q95 − Q05'], scales.spread_change),
    '|S/N|': (abs_sn_final_da, scales.sn),
}

In [ ]:
#(t): The share beyond each end (%)
saturation_df = 100 * pd.DataFrame({name: fr.saturation(da, scale) for name, (da, scale) in range_checks.items()}).T
saturation_df.round(1)

## Main Figures

**Figure 1. Historical change in the distribution of winter (JJA) near-surface air temperature.** Multi-model median of (a–d) hist-nat and (e–h) historical, both over 1993–2013, and (i–l) the change between them: the 5th percentile, the mean, the 95th percentile, and the width Q95 − Q05 of the internal variability (each member minus its ensemble's forced response). Dots: fewer than two-thirds of the models have a significant change of one sign (the hist-nat bootstrap: 10 random hist-nat members over the same years; p < 0.05). After Bracegirdle et al. (2024), Fig. 1.

In [ ]:
#(t): The models' median in WINTER: hist-nat's and historical's Q05, mean and Q95 (one colour bar) and width (another),
#(t): then the change, dotted where not robust
panel_size = antarctica.panel_size_to_fit(n_rows=3, n_cols=4, n_colorbar_rows=2, n_legends=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=4, n_colorbar_rows=2, n_legends=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=3, n_cols=4)
    #(c): Each colour bar under the maps it is for: hist-nat's and historical's between rows 2 and 3 (Q05, the mean
    #(c): and Q95 under columns 1-3, the width under column 4), the change's under row 3
    temperature_cax = antarctica.colorbar_axis_below(axes[:2, :3])
    spread_cax = antarctica.colorbar_axis_below(axes[:2, 3])
    change_cax = antarctica.colorbar_axis_below(axes[2, :3])
    spread_change_cax = antarctica.colorbar_axis_below(axes[2, 3])

    #(c): Rows 1 and 2: hist-nat and historical; Q05, the mean and Q95 on one colour bar, the width on another
    season_hist_nat_ds = median_hist_nat_fields_ds.sel(season=WINTER)
    season_historical_ds = median_experiment_fields_ds.sel(experiment='historical', season=WINTER)
    for row, row_ds in enumerate([season_hist_nat_ds, season_historical_ds]):
        antarctica.heatmaps_of_antarctica(row_ds[['Q05', 'Mean', 'Q95']].to_dataarray('quantity'), None, 'quantity',
                                          axes=axes[row, :3], cax=temperature_cax if row == 0 else None,
                                          **scales.temperature.colours())
        antarctica.heatmaps_of_antarctica(row_ds['Q95 − Q05'], None, None, axes=axes[row, 3],
                                          cax=spread_cax if row == 0 else None, **scales.spread.colours())
    #(c): Row 3: the change, on the change's colour bars; the mean and the width dotted where not robust
    season_change_ds = median_change_fields_ds.sel(experiment='historical', season=WINTER)
    season_not_robust_ds = not_robust_ds.sel(experiment='historical', season=WINTER)
    antarctica.heatmaps_of_antarctica(season_change_ds[['Q05', 'Mean', 'Q95']].to_dataarray('quantity'), None, 'quantity',
                                      axes=axes[2, :3], cax=change_cax, **scales.change.colours())
    spread_change_cbar = antarctica.heatmaps_of_antarctica(season_change_ds['Q95 − Q05'], None, None, axes=axes[2, 3],
                                                           cax=spread_change_cax, **scales.spread_change.colours())
    #(c): Every other tick, so they fit under one map
    spread_change_cbar.set_ticks(scales.spread_change.levels[::2])
    antarctica.not_significant_dots(season_not_robust_ds['Mean'], ax=axes[2, 1])
    antarctica.not_significant_dots(season_not_robust_ds['Q95 − Q05'], ax=axes[2, 3])

    antarctica.label_columns(axes, ['Q05', 'Mean', 'Q95', 'Q95 − Q05'], fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['hist-nat', 'historical', 'Change'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
    fig.legend(handles=[not_robust_handle], loc='outside lower center', frameon=False, fontsize=STYLE.cbar_label_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'fig1_distribution_historical_{WINTER}')

**Figure 2. How each forcing changes the distribution of winter (JJA) temperature.** Multi-model median change, 1993–2013 against hist-nat, of the 5th percentile, the mean, the 95th percentile and the width (Q95 − Q05) of the internal variability, for each single forcing (rows) and all forcings together (historical). Dots as in Fig. 1.

In [ ]:
#(t): The changes in WINTER, every forcing
winter_changes_ds = median_change_fields_ds.sel(experiment=FORCED_EXPERIMENTS, season=WINTER)
winter_changes_ds

In [ ]:
#(t): and where they are not robust
winter_not_robust_ds = not_robust_ds.sel(experiment=FORCED_EXPERIMENTS, season=WINTER)
winter_not_robust_ds

In [ ]:
#(t): Each forcing's median change in WINTER (rows): Q05, the mean and Q95 (one colour bar), and the width,
#(t): dotted where not robust
panel_size = antarctica.panel_size_to_fit(n_rows=len(FORCED_EXPERIMENTS), n_cols=4, n_colorbar_rows=1, n_legends=1,
                                          width=WIDTH, height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=4, n_colorbar_rows=1,
                                                        n_legends=1, panel_size=panel_size,
                                                        title_height=PAPER_TITLE_HEIGHT), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=4)
    #(c): Side by side under the last row: the change's colour bar under columns 1-3, the width's under column 4
    change_cax = antarctica.colorbar_axis_below(axes[:, :3])
    spread_change_cax = antarctica.colorbar_axis_below(axes[:, 3])

    antarctica.heatmaps_of_antarctica(winter_changes_ds[['Q05', 'Mean', 'Q95']].to_dataarray('quantity'), 'experiment', 'quantity',
                                      axes=axes[:, :3], cax=change_cax, **scales.change.colours())
    spread_change_cbar = antarctica.heatmaps_of_antarctica(winter_changes_ds['Q95 − Q05'], 'experiment', None,
                                                           axes=axes[:, 3], cax=spread_change_cax,
                                                           **scales.spread_change.colours())
    #(c): Every other tick, so they fit under one column
    spread_change_cbar.set_ticks(scales.spread_change.levels[::2])
    antarctica.not_significant_dots_grid(winter_not_robust_ds['Mean'], 'experiment', None, axes=axes[:, 1])
    antarctica.not_significant_dots_grid(winter_not_robust_ds['Q95 − Q05'], 'experiment', None, axes=axes[:, 3])

    antarctica.label_columns(axes, ['ΔQ05', 'Δ mean', 'ΔQ95', 'Δ(Q95 − Q05)'], fontsize=STYLE.label_size)
    antarctica.label_rows(axes, FORCED_EXPERIMENTS, fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
    fig.legend(handles=[not_robust_handle], loc='outside lower center', frameon=False, fontsize=STYLE.cbar_label_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'fig2_forcings_{WINTER}')

**Figure 3. As Fig. 2, in summer (DJF).**

In [ ]:
#(t): The changes in SUMMER, every forcing
summer_changes_ds = median_change_fields_ds.sel(experiment=FORCED_EXPERIMENTS, season=SUMMER)
summer_changes_ds

In [ ]:
#(t): and where they are not robust
summer_not_robust_ds = not_robust_ds.sel(experiment=FORCED_EXPERIMENTS, season=SUMMER)
summer_not_robust_ds

In [ ]:
#(t): Each forcing's median change in SUMMER (rows): Q05, the mean and Q95 (one colour bar), and the width,
#(t): dotted where not robust
panel_size = antarctica.panel_size_to_fit(n_rows=len(FORCED_EXPERIMENTS), n_cols=4, n_colorbar_rows=1, n_legends=1,
                                          width=WIDTH, height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=4, n_colorbar_rows=1,
                                                        n_legends=1, panel_size=panel_size,
                                                        title_height=PAPER_TITLE_HEIGHT), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=4)
    #(c): Side by side under the last row: the change's colour bar under columns 1-3, the width's under column 4
    change_cax = antarctica.colorbar_axis_below(axes[:, :3])
    spread_change_cax = antarctica.colorbar_axis_below(axes[:, 3])

    antarctica.heatmaps_of_antarctica(summer_changes_ds[['Q05', 'Mean', 'Q95']].to_dataarray('quantity'), 'experiment', 'quantity',
                                      axes=axes[:, :3], cax=change_cax, **scales.change.colours())
    spread_change_cbar = antarctica.heatmaps_of_antarctica(summer_changes_ds['Q95 − Q05'], 'experiment', None,
                                                           axes=axes[:, 3], cax=spread_change_cax,
                                                           **scales.spread_change.colours())
    #(c): Every other tick, so they fit under one column
    spread_change_cbar.set_ticks(scales.spread_change.levels[::2])
    antarctica.not_significant_dots_grid(summer_not_robust_ds['Mean'], 'experiment', None, axes=axes[:, 1])
    antarctica.not_significant_dots_grid(summer_not_robust_ds['Q95 − Q05'], 'experiment', None, axes=axes[:, 3])

    antarctica.label_columns(axes, ['ΔQ05', 'Δ mean', 'ΔQ95', 'Δ(Q95 − Q05)'], fontsize=STYLE.label_size)
    antarctica.label_rows(axes, FORCED_EXPERIMENTS, fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
    fig.legend(handles=[not_robust_handle], loc='outside lower center', frameon=False, fontsize=STYLE.cbar_label_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'fig3_forcings_{SUMMER}')

**Figure 4. Where the mean and the variability have changed together.** For each forcing (columns) in summer (DJF) and winter (JJA): the robust change in the mean (rows of the key: lower, none, higher) combined with the robust change in the width of the internal variability (columns of the key: narrower, none, wider). A change is robust where at least two-thirds of the models have a significant change of that sign (tests as in Fig. 1).

In [ ]:
#(t): The robust joint class, every forcing, in summer and winter
seasons_class_da = robust_joint_class_da.sel(experiment=FORCED_EXPERIMENTS, season=[SUMMER, WINTER])
seasons_class_da

In [ ]:
#(t): The robust joint change in the mean and the width: summer and winter (rows), every forcing (columns), and its key
panel_size = antarctica.panel_size_to_fit(n_rows=len(seasons_class_da.season), n_cols=len(FORCED_EXPERIMENTS),
                                          n_colorbar_rows=0, n_keys=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(seasons_class_da.season),
                                                        n_cols=len(FORCED_EXPERIMENTS),
                                                        n_colorbar_rows=0, n_keys=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(seasons_class_da.season), n_cols=len(FORCED_EXPERIMENTS))
    key_ax = antarctica.key_axis_below(axes)

    antarctica.heatmaps_of_antarctica(seasons_class_da, 'season', 'experiment', axes=axes, **scales.joint.colours())
    antarctica.bivariate_key(key_ax, scales.joint.cmap, *scales.joint.key)

    antarctica.label_columns(axes, seasons_class_da['experiment'].values, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, seasons_class_da['season'].values, fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, PAPER_DIR, 'fig4_joint_change')

**Figure 5. The change in the mean and in the width, averaged south of 60°S.** Each model's area-mean change (markers; colour: forcing) in each season, 1993–2013 against hist-nat; bars: the multi-model median. Top: the mean; bottom: the width Q95 − Q05 of the internal variability.

In [ ]:
#(t): Area means south of 60°S of every model's changes
change_regional_ds = rc.regional_mean(change_ds.sel(experiment=FORCED_EXPERIMENTS), lat_max=-60, mean_test=MEAN_TEST)
change_regional_ds

In [ ]:
#(t): Every model's area-mean change (markers) and the multi-model median (bars) for each forcing (its colour), the
#(t): mean (top) and the width (bottom), seasons across, a page wide
with plt.rc_context(STYLE.rc):
    panel_width = charts.panel_width_to_fit(len(SEASONS), WIDTH)
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=2, n_cols=len(SEASONS), n_legends=1, panel_width=panel_width,
                                                      panel_height=1.5, top_margin=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = fig.add_gridspec(2, len(SEASONS)).subplots(sharex=True, sharey='row', squeeze=False)
    charts.model_dots_grid(change_regional_ds['mean_change'].sel(season=SEASONS), 'experiment', None, 'season',
                           axes=axes[0], color=FORCING_COLORS)
    charts.model_dots_grid(change_regional_ds['width_change'].sel(season=SEASONS), 'experiment', None, 'season',
                           axes=axes[1], color=FORCING_COLORS)
    axes[0, 0].set_ylabel(f'Δ mean ({VARIABLE.units})')
    axes[1, 0].set_ylabel(f'Δ(Q95 − Q05) ({VARIABLE.units})')
    for ax, season in zip(axes[0], SEASONS):
        ax.set_title(season, loc='left')
    #(c): The forcings' names turned, so four fit under a narrow panel
    for ax in axes[-1]:
        for label in ax.get_xticklabels():
            label.set(rotation=40, ha='right')
    fig.legend(handles=charts.model_legend_handles(change_regional_ds['model'].values)
               + [Patch(color='0.6', alpha=0.5, label='Multi-model median')], loc='outside lower center',
               ncols=change_regional_ds.sizes['model'] + 1, frameon=False)
    charts.tag_panels(axes)

In [ ]:
fr.save(fig, PAPER_DIR, 'fig5_area_means')

**Figure 6. The change at each latitude in winter (JJA).** Zonal-mean change, 1993–2013 against hist-nat, for each forcing (rows) and every model (lines): the mean; the low extremes Δ(Q05 − Q50) and the high extremes Δ(Q95 − Q50) of the internal variability, each measured from the median; and the width Δ(Q95 − Q05), the high minus the low. Dots: significant zonal-mean change in that model (the hist-nat bootstrap; p < 0.05). Dashed: the sea-ice edge (equivalent latitude) of each model in 1993–2013. After Bracegirdle et al. (2024), Fig. 2.

In [ ]:
#(t): The four columns, zonally averaged, with the zonal tests
zonal_ds = zonal.zonal_summary(change_ds, zonal_mean_ds, zonal_width_ds)
zonal_ds

In [ ]:
#(t): The zonal-mean change in WINTER, each forcing (rows) and every model (lines, its shade of the forcing's colour and its line style), the four changes
#(t): across; a dot where the mean's or the width's change is significant; each model's ice edge dashed. The extremes
#(t): and the width share one scale
zonal_columns = list(rc.EXTREMES)
zonal_models = list(zonal_ds['model'].values)
with plt.rc_context(STYLE.rc):
    panel_width = charts.panel_width_to_fit(len(zonal_columns), WIDTH)
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(zonal_columns),
                                                      n_legends=1,
                                                      panel_width=panel_width, panel_height=1.15, top_margin=0.6),
                     layout='constrained')
    axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(zonal_columns)).subplots(sharex=True, sharey='col',
                                                                                  squeeze=False)
    for ax in axes.ravel():
        ax.axhline(0, color='0.64', linewidth=0.8, zorder=1)
    for i, experiment in enumerate(FORCED_EXPERIMENTS):
        model_styles = charts.model_lines(experiment, zonal_models)
        for model in zonal_models:
            model_zonal_ds = zonal_ds.sel(model=model, season=WINTER, experiment=experiment)
            model_color = model_styles[model]['color']
            charts.lines_grid(model_zonal_ds[zonal_columns].to_dataarray('column'), 'lat', None, 'column',
                              axes=axes[i], **model_styles[model])
            charts.plot_significant_dots(model_zonal_ds['mean_change'], model_zonal_ds['mean_pvalue'] < ALPHA, 'lat',
                                         ax=axes[i, 0], color=model_color)
            charts.plot_significant_dots(model_zonal_ds['width_change'], model_zonal_ds['width_pvalue'] < ALPHA, 'lat',
                                         ax=axes[i, 3], color=model_color)
            if ice_edges_da is not None:
                zp.ice_edge_lines_grid(ice_edges_da.sel(model=model, experiment=experiment, season=WINTER).expand_dims(
                    column=zonal_columns), None, 'column', axes=axes[i], color=model_color)

    for ax, column in zip(axes[0], zonal_columns):
        ax.set_title(f'{rcp.EXTREME_TITLES[column]}\n{rc.EXTREMES[column]}', loc='left')
    for ax, experiment in zip(axes[:, 0], FORCED_EXPERIMENTS):
        ax.set_ylabel(f'{experiment}\n({VARIABLE.units})')
    for ax in axes[-1]:
        ax.set_xlabel('Latitude')
    axes[0, 0].xaxis.set_major_formatter(zp.latitude_label)
    axes[0, 0].set_xlim(float(zonal_ds['lat'].min()), float(zonal_ds['lat'].max()))
    extremes_ylim = (min(ax.get_ylim()[0] for ax in axes[0, 1:]), max(ax.get_ylim()[1] for ax in axes[0, 1:]))
    for ax in axes[0, 1:]:
        ax.set_ylim(extremes_ylim)
    #(c): One legend for every row: model k is the k-th shade of each row's colour, shown in historical's greys
    fig.legend(handles=[Line2D([], [], **style)
                        for style in charts.model_lines('historical', zonal_models).values()] + [
Line2D([], [], linestyle='none', marker='o', markersize=4, color='k', markeredgecolor='white',
    label=f'Significant (p < {ALPHA:g})'),
Line2D([], [], color='k', linewidth=1.0, linestyle=zp.EDGE_LINESTYLE, label='Sea-ice edge')],
loc='outside lower center', ncols=len(zonal_models) + 2, frameon=False)
    charts.tag_panels(axes)

In [ ]:
fr.save(fig, PAPER_DIR, f'fig6_zonal_{WINTER}')

## Supplementary Figures

**Figure S1. As Fig. 1, in summer (DJF).**

In [ ]:
#(t): The models' median in SUMMER: hist-nat's and historical's Q05, mean and Q95 (one colour bar) and width (another),
#(t): then the change, dotted where not robust
panel_size = antarctica.panel_size_to_fit(n_rows=3, n_cols=4, n_colorbar_rows=2, n_legends=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=4, n_colorbar_rows=2, n_legends=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=3, n_cols=4)
    #(c): Each colour bar under the maps it is for: hist-nat's and historical's between rows 2 and 3 (Q05, the mean
    #(c): and Q95 under columns 1-3, the width under column 4), the change's under row 3
    temperature_cax = antarctica.colorbar_axis_below(axes[:2, :3])
    spread_cax = antarctica.colorbar_axis_below(axes[:2, 3])
    change_cax = antarctica.colorbar_axis_below(axes[2, :3])
    spread_change_cax = antarctica.colorbar_axis_below(axes[2, 3])

    #(c): Rows 1 and 2: hist-nat and historical; Q05, the mean and Q95 on one colour bar, the width on another
    season_hist_nat_ds = median_hist_nat_fields_ds.sel(season=SUMMER)
    season_historical_ds = median_experiment_fields_ds.sel(experiment='historical', season=SUMMER)
    for row, row_ds in enumerate([season_hist_nat_ds, season_historical_ds]):
        antarctica.heatmaps_of_antarctica(row_ds[['Q05', 'Mean', 'Q95']].to_dataarray('quantity'), None, 'quantity',
                                          axes=axes[row, :3], cax=temperature_cax if row == 0 else None,
                                          **scales.temperature.colours())
        antarctica.heatmaps_of_antarctica(row_ds['Q95 − Q05'], None, None, axes=axes[row, 3],
                                          cax=spread_cax if row == 0 else None, **scales.spread.colours())
    #(c): Row 3: the change, on the change's colour bars; the mean and the width dotted where not robust
    season_change_ds = median_change_fields_ds.sel(experiment='historical', season=SUMMER)
    season_not_robust_ds = not_robust_ds.sel(experiment='historical', season=SUMMER)
    antarctica.heatmaps_of_antarctica(season_change_ds[['Q05', 'Mean', 'Q95']].to_dataarray('quantity'), None, 'quantity',
                                      axes=axes[2, :3], cax=change_cax, **scales.change.colours())
    spread_change_cbar = antarctica.heatmaps_of_antarctica(season_change_ds['Q95 − Q05'], None, None, axes=axes[2, 3],
                                                           cax=spread_change_cax, **scales.spread_change.colours())
    #(c): Every other tick, so they fit under one map
    spread_change_cbar.set_ticks(scales.spread_change.levels[::2])
    antarctica.not_significant_dots(season_not_robust_ds['Mean'], ax=axes[2, 1])
    antarctica.not_significant_dots(season_not_robust_ds['Q95 − Q05'], ax=axes[2, 3])

    antarctica.label_columns(axes, ['Q05', 'Mean', 'Q95', 'Q95 − Q05'], fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['hist-nat', 'historical', 'Change'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
    fig.legend(handles=[not_robust_handle], loc='outside lower center', frameon=False, fontsize=STYLE.cbar_label_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS1_distribution_historical_{SUMMER}')

**Figure S2. Every model's historical change in the distribution of winter (JJA) temperature.** As the bottom row of Fig. 1, for each model (rows), with each model's joint class (as in Fig. 4, from that model's own tests). Dots: not significant in that model (p ≥ 0.05).

In [ ]:
#(t): Every model, historical, WINTER: the changes
winter_model_changes_ds = change_fields_ds.sel(experiment='historical', season=WINTER)
winter_model_changes_ds

In [ ]:
#(t): where they are not significant
winter_model_not_significant_ds = change_not_significant_ds.sel(experiment='historical', season=WINTER)
winter_model_not_significant_ds

In [ ]:
#(t): and the joint class
winter_model_class_da = model_joint_class_da.sel(experiment='historical', season=WINTER)
winter_model_class_da

In [ ]:
#(t): Every model's historical change (rows): Q05, the mean and Q95 (one colour bar), the width, and what changed
#(t): (its key); the mean and the width dotted where not significant
panel_size = antarctica.panel_size_to_fit(n_rows=winter_model_changes_ds.sizes['model'], n_cols=5, n_colorbar_rows=0,
                                          n_keys=1, n_legends=1, width=WIDTH, height=HEIGHT,
                                          title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=winter_model_changes_ds.sizes['model'], n_cols=5,
                                                        n_colorbar_rows=0, n_keys=1, n_legends=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=winter_model_changes_ds.sizes['model'], n_cols=5)
    #(c): Side by side under the last row: the change's colour bar under columns 1-3, the width's under column 4 and
    #(c): the key under column 5 (the row is the key's, so n_colorbar_rows=0)
    change_cax = antarctica.colorbar_axis_below(axes[:, :3])
    spread_change_cax = antarctica.colorbar_axis_below(axes[:, 3])
    key_ax = antarctica.key_axis_below(axes[:, 4])

    antarctica.heatmaps_of_antarctica(winter_model_changes_ds[['Q05', 'Mean', 'Q95']].to_dataarray('quantity'), 'model', 'quantity',
                                      axes=axes[:, :3], cax=change_cax, **scales.change.colours())
    spread_change_cbar = antarctica.heatmaps_of_antarctica(winter_model_changes_ds['Q95 − Q05'], 'model', None,
                                                           axes=axes[:, 3], cax=spread_change_cax,
                                                           **scales.spread_change.colours())
    #(c): Every fourth tick (-1, 0, 1), so they fit under one narrow column
    spread_change_cbar.set_ticks(scales.spread_change.levels[::4])
    antarctica.heatmaps_of_antarctica(winter_model_class_da, 'model', None, axes=axes[:, 4], **scales.joint.colours())
    antarctica.bivariate_key(key_ax, scales.joint.cmap, *scales.joint.key)
    antarctica.not_significant_dots_grid(winter_model_not_significant_ds['Mean'], 'model', None, axes=axes[:, 1])
    antarctica.not_significant_dots_grid(winter_model_not_significant_ds['Q95 − Q05'], 'model', None, axes=axes[:, 3])

    antarctica.label_columns(axes, ['ΔQ05', 'Δ mean', 'ΔQ95', 'Δ(Q95 − Q05)', 'What changed'], fontsize=STYLE.label_size)
    antarctica.label_rows(axes, winter_model_changes_ds['model'].values, fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
    fig.legend(handles=[not_significant_handle], loc='outside lower center', frameon=False,
               fontsize=STYLE.cbar_label_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS2_models_historical_{WINTER}')

**Figure S3. As Fig. S2, in summer (DJF).**

In [ ]:
#(t): Every model, historical, SUMMER: the changes
summer_model_changes_ds = change_fields_ds.sel(experiment='historical', season=SUMMER)
summer_model_changes_ds

In [ ]:
#(t): where they are not significant
summer_model_not_significant_ds = change_not_significant_ds.sel(experiment='historical', season=SUMMER)
summer_model_not_significant_ds

In [ ]:
#(t): and the joint class
summer_model_class_da = model_joint_class_da.sel(experiment='historical', season=SUMMER)
summer_model_class_da

In [ ]:
#(t): Every model's historical change (rows): Q05, the mean and Q95 (one colour bar), the width, and what changed
#(t): (its key); the mean and the width dotted where not significant
panel_size = antarctica.panel_size_to_fit(n_rows=summer_model_changes_ds.sizes['model'], n_cols=5, n_colorbar_rows=0,
                                          n_keys=1, n_legends=1, width=WIDTH, height=HEIGHT,
                                          title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=summer_model_changes_ds.sizes['model'], n_cols=5,
                                                        n_colorbar_rows=0, n_keys=1, n_legends=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=summer_model_changes_ds.sizes['model'], n_cols=5)
    #(c): Side by side under the last row: the change's colour bar under columns 1-3, the width's under column 4 and
    #(c): the key under column 5 (the row is the key's, so n_colorbar_rows=0)
    change_cax = antarctica.colorbar_axis_below(axes[:, :3])
    spread_change_cax = antarctica.colorbar_axis_below(axes[:, 3])
    key_ax = antarctica.key_axis_below(axes[:, 4])

    antarctica.heatmaps_of_antarctica(summer_model_changes_ds[['Q05', 'Mean', 'Q95']].to_dataarray('quantity'), 'model', 'quantity',
                                      axes=axes[:, :3], cax=change_cax, **scales.change.colours())
    spread_change_cbar = antarctica.heatmaps_of_antarctica(summer_model_changes_ds['Q95 − Q05'], 'model', None,
                                                           axes=axes[:, 3], cax=spread_change_cax,
                                                           **scales.spread_change.colours())
    #(c): Every fourth tick (-1, 0, 1), so they fit under one narrow column
    spread_change_cbar.set_ticks(scales.spread_change.levels[::4])
    antarctica.heatmaps_of_antarctica(summer_model_class_da, 'model', None, axes=axes[:, 4], **scales.joint.colours())
    antarctica.bivariate_key(key_ax, scales.joint.cmap, *scales.joint.key)
    antarctica.not_significant_dots_grid(summer_model_not_significant_ds['Mean'], 'model', None, axes=axes[:, 1])
    antarctica.not_significant_dots_grid(summer_model_not_significant_ds['Q95 − Q05'], 'model', None, axes=axes[:, 3])

    antarctica.label_columns(axes, ['ΔQ05', 'Δ mean', 'ΔQ95', 'Δ(Q95 − Q05)', 'What changed'], fontsize=STYLE.label_size)
    antarctica.label_rows(axes, summer_model_changes_ds['model'].values, fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)
    fig.legend(handles=[not_significant_handle], loc='outside lower center', frameon=False,
               fontsize=STYLE.cbar_label_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS3_models_historical_{SUMMER}')

**Figures S4–S7. How many models have a significant change.** At each grid point, the number of models with a significant increase (top rows, red) and decrease (bottom rows, blue) in the mean (S4: DJF, S5: JJA) and in the width Q95 − Q05 (S6: DJF, S7: JJA), for each forcing. Tests as in Fig. 1.

In [ ]:
#(t): The counts: the mean
mean_counts_da = sig.significant_counts(change_fields_ds['Mean'], change_significant_ds['Mean'])

In [ ]:
#(t): and the width
width_counts_da = sig.significant_counts(change_fields_ds['Q95 − Q05'], change_significant_ds['Q95 − Q05'])

In [ ]:
#(t): The decreases as negative counts, so that one colour bar (blues for decreases, reds for increases) serves both
#(t): rows of counts, in the mean and in the width
signed_mean_counts_da = xr.concat([mean_counts_da.sel(direction=['increase']), -mean_counts_da.sel(direction=['decrease'])],
                                  dim='direction')
signed_width_counts_da = xr.concat([width_counts_da.sel(direction=['increase']),
                                    -width_counts_da.sel(direction=['decrease'])], dim='direction')
signed_width_counts_da

In [ ]:
#(t): How many models: a significant increase and decrease (rows) in SUMMER, every forcing (columns)
panel_size = antarctica.panel_size_to_fit(n_rows=2, n_cols=len(FORCED_EXPERIMENTS), n_colorbar_rows=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(FORCED_EXPERIMENTS), n_colorbar_rows=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(FORCED_EXPERIMENTS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(signed_mean_counts_da.sel(experiment=FORCED_EXPERIMENTS, season=SUMMER),
                                      'direction', 'experiment',
                                      axes=axes, cax=cax,
                                      **fr.count_scale(N_MODELS, 'Models with a significant decrease (blue) or increase (red) in the mean').colours())

    antarctica.label_columns(axes, FORCED_EXPERIMENTS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['Increase', 'Decrease'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS4_mean_counts_{SUMMER}')

In [ ]:
#(t): How many models: a significant increase and decrease (rows) in WINTER, every forcing (columns)
panel_size = antarctica.panel_size_to_fit(n_rows=2, n_cols=len(FORCED_EXPERIMENTS), n_colorbar_rows=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(FORCED_EXPERIMENTS), n_colorbar_rows=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(FORCED_EXPERIMENTS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(signed_mean_counts_da.sel(experiment=FORCED_EXPERIMENTS, season=WINTER),
                                      'direction', 'experiment',
                                      axes=axes, cax=cax,
                                      **fr.count_scale(N_MODELS, 'Models with a significant decrease (blue) or increase (red) in the mean').colours())

    antarctica.label_columns(axes, FORCED_EXPERIMENTS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['Increase', 'Decrease'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS5_mean_counts_{WINTER}')

In [ ]:
#(t): How many models: a significant increase and decrease (rows) in SUMMER, every forcing (columns)
panel_size = antarctica.panel_size_to_fit(n_rows=2, n_cols=len(FORCED_EXPERIMENTS), n_colorbar_rows=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(FORCED_EXPERIMENTS), n_colorbar_rows=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(FORCED_EXPERIMENTS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(signed_width_counts_da.sel(experiment=FORCED_EXPERIMENTS, season=SUMMER),
                                      'direction', 'experiment',
                                      axes=axes, cax=cax,
                                      **fr.count_scale(N_MODELS, 'Models significantly narrower (blue) or wider (red)').colours())

    antarctica.label_columns(axes, FORCED_EXPERIMENTS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['Wider', 'Narrower'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS6_width_counts_{SUMMER}')

In [ ]:
#(t): How many models: a significant increase and decrease (rows) in WINTER, every forcing (columns)
panel_size = antarctica.panel_size_to_fit(n_rows=2, n_cols=len(FORCED_EXPERIMENTS), n_colorbar_rows=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2, n_cols=len(FORCED_EXPERIMENTS), n_colorbar_rows=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2, n_cols=len(FORCED_EXPERIMENTS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(signed_width_counts_da.sel(experiment=FORCED_EXPERIMENTS, season=WINTER),
                                      'direction', 'experiment',
                                      axes=axes, cax=cax,
                                      **fr.count_scale(N_MODELS, 'Models significantly narrower (blue) or wider (red)').colours())

    antarctica.label_columns(axes, FORCED_EXPERIMENTS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, ['Wider', 'Narrower'], fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS7_width_counts_{WINTER}')

**Figures S8, S9. Signal to noise.** |S/N| in 2003 (the centre of 1993–2013) for every model (rows) and forcing (columns), in DJF (S8) and JJA (S9): the forced response minus hist-nat's, over the pooled year-to-year standard deviation, whichever way the shift goes. It measures how far apart the two distributions have moved: yellow below 1, orange from 1 to 2, red above 2.

In [ ]:
#(t): |S/N| in SUMMER: every model (rows) and forcing (columns)
panel_size = antarctica.panel_size_to_fit(n_rows=abs_sn_final_da.sizes['model'], n_cols=len(FORCED_EXPERIMENTS),
                                          n_colorbar_rows=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=abs_sn_final_da.sizes['model'],
                                                        n_cols=len(FORCED_EXPERIMENTS),
                                                        n_colorbar_rows=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=abs_sn_final_da.sizes['model'], n_cols=len(FORCED_EXPERIMENTS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(abs_sn_final_da.sel(experiment=FORCED_EXPERIMENTS, season=SUMMER), 'model',
                                      'experiment',
                                      axes=axes, cax=cax, **scales.sn.colours())

    antarctica.label_columns(axes, FORCED_EXPERIMENTS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, abs_sn_final_da['model'].values, fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS8_sn_{SUMMER}')

In [ ]:
#(t): |S/N| in WINTER: every model (rows) and forcing (columns)
panel_size = antarctica.panel_size_to_fit(n_rows=abs_sn_final_da.sizes['model'], n_cols=len(FORCED_EXPERIMENTS),
                                          n_colorbar_rows=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=abs_sn_final_da.sizes['model'],
                                                        n_cols=len(FORCED_EXPERIMENTS),
                                                        n_colorbar_rows=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=abs_sn_final_da.sizes['model'], n_cols=len(FORCED_EXPERIMENTS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(abs_sn_final_da.sel(experiment=FORCED_EXPERIMENTS, season=WINTER), 'model',
                                      'experiment',
                                      axes=axes, cax=cax, **scales.sn.colours())

    antarctica.label_columns(axes, FORCED_EXPERIMENTS, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, abs_sn_final_da['model'].values, fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS9_sn_{WINTER}')

**Figure S10. The change at every percentile.** Area mean (south of 40°S) of the change in every 5th percentile, 1993–2013 against hist-nat over the same years, every model (lines), in DJF (top) and JJA (bottom). A flat line is a pure shift; a rising line, a widening distribution.

In [ ]:
#(t): The area-mean change at every 5th percentile in SUMMER, every model (lines, its shade of the forcing's colour and its line style), one panel per forcing
summer_curve_da = quantile_curve_da.sel(experiment=FORCED_EXPERIMENTS, season=SUMMER)
summer_curve_da = summer_curve_da.assign_coords(
    percentile=('quantile', 100 * summer_curve_da['quantile'].values)).swap_dims(quantile='percentile')
curve_models = list(summer_curve_da['model'].values)
with plt.rc_context(STYLE.rc):
    panel_width = charts.panel_width_to_fit(len(FORCED_EXPERIMENTS), WIDTH)
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(FORCED_EXPERIMENTS), n_legends=1,
                                                      panel_width=panel_width, panel_height=1.4), layout='constrained')
    axes = fig.add_gridspec(1, len(FORCED_EXPERIMENTS)).subplots(sharex=True, sharey=True, squeeze=False)
    for ax in axes[0]:
        ax.axhline(0, color='0.6', linewidth=0.8)
    for j, experiment in enumerate(FORCED_EXPERIMENTS):
        model_styles = charts.model_lines(experiment, curve_models)
        for k, model in enumerate(curve_models):
            charts.plot_lines(summer_curve_da.sel(experiment=experiment, model=model), 'percentile', ax=axes[0, j],
                              marker=charts.MODEL_MARKERS[k], markersize=2.8, **model_styles[model])
    axes[0, 0].set_xticks([0, 50, 100])
    for ax, experiment in zip(axes[0], FORCED_EXPERIMENTS):
        ax.set_title(experiment, loc='left')
        ax.set_xlabel('Percentile')
    axes[0, 0].set_ylabel(f'Change ({VARIABLE.units})')
    #(c): One legend for every panel: model k is the k-th shade of each panel's colour, shown in historical's greys
    legend_styles = charts.model_lines('historical', curve_models)
    fig.legend(handles=[Line2D([], [], marker=charts.MODEL_MARKERS[k], markersize=2.8,
                               **legend_styles[model]) for k, model in enumerate(curve_models)],
               loc='outside lower center', ncols=len(curve_models), frameon=False)
    fig.suptitle(SUMMER, x=0.01, ha='left')

In [ ]:
fr.save(fig, PAPER_DIR, f'figS10a_quantile_curves_{SUMMER}')

In [ ]:
#(t): ... in WINTER
winter_curve_da = quantile_curve_da.sel(experiment=FORCED_EXPERIMENTS, season=WINTER)
winter_curve_da = winter_curve_da.assign_coords(
    percentile=('quantile', 100 * winter_curve_da['quantile'].values)).swap_dims(quantile='percentile')
with plt.rc_context(STYLE.rc):
    panel_width = charts.panel_width_to_fit(len(FORCED_EXPERIMENTS), WIDTH)
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(FORCED_EXPERIMENTS), n_legends=1,
                                                      panel_width=panel_width, panel_height=1.4), layout='constrained')
    axes = fig.add_gridspec(1, len(FORCED_EXPERIMENTS)).subplots(sharex=True, sharey=True, squeeze=False)
    for ax in axes[0]:
        ax.axhline(0, color='0.6', linewidth=0.8)
    for j, experiment in enumerate(FORCED_EXPERIMENTS):
        model_styles = charts.model_lines(experiment, curve_models)
        for k, model in enumerate(curve_models):
            charts.plot_lines(winter_curve_da.sel(experiment=experiment, model=model), 'percentile', ax=axes[0, j],
                              marker=charts.MODEL_MARKERS[k], markersize=2.8, **model_styles[model])
    axes[0, 0].set_xticks([0, 50, 100])
    for ax, experiment in zip(axes[0], FORCED_EXPERIMENTS):
        ax.set_title(experiment, loc='left')
        ax.set_xlabel('Percentile')
    axes[0, 0].set_ylabel(f'Change ({VARIABLE.units})')
    #(c): One legend for every panel: model k is the k-th shade of each panel's colour, shown in historical's greys
    legend_styles = charts.model_lines('historical', curve_models)
    fig.legend(handles=[Line2D([], [], marker=charts.MODEL_MARKERS[k], markersize=2.8,
                               **legend_styles[model]) for k, model in enumerate(curve_models)],
               loc='outside lower center', ncols=len(curve_models), frameon=False)
    fig.suptitle(WINTER, x=0.01, ha='left')

In [ ]:
fr.save(fig, PAPER_DIR, f'figS10b_quantile_curves_{WINTER}')

**Figure S11. Additivity.** For each model (rows), winter (JJA): the sum of the four single-forcing changes (hist-nat, hist-GHG, hist-aer, hist-totalO3), the historical change, and the residual (historical minus the sum), each experiment's final 21 years against its own 1850–1900.

In [ ]:
#(t): Every term: each part, their sum, historical and the residual
additivity_ds = rc.additivity(own_changes_da)

In [ ]:
#(t): The sum, historical and the residual, every model, WINTER
winter_additivity_da = additivity_ds['response'].sel(season=WINTER, term=['sum of parts', 'historical', 'residual'])
winter_additivity_da

In [ ]:
#(t): Additivity in winter: every model (rows), the sum of the parts, historical and the residual (columns)
panel_size = antarctica.panel_size_to_fit(n_rows=winter_additivity_da.sizes['model'],
                                          n_cols=winter_additivity_da.sizes['term'], n_colorbar_rows=1, width=WIDTH,
                                          height=HEIGHT, title_height=PAPER_TITLE_HEIGHT)
with plt.rc_context(STYLE.rc):
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=winter_additivity_da.sizes['model'],
                                                        n_cols=winter_additivity_da.sizes['term'], n_colorbar_rows=1,
                                                        panel_size=panel_size, title_height=PAPER_TITLE_HEIGHT),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=winter_additivity_da.sizes['model'],
                               n_cols=winter_additivity_da.sizes['term'])
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(winter_additivity_da, 'model', 'term', axes=axes, cax=cax,
                                      **scales.change.colours())

    antarctica.label_columns(axes, winter_additivity_da['term'].values, fontsize=STYLE.label_size)
    antarctica.label_rows(axes, winter_additivity_da['model'].values, fontsize=STYLE.label_size)
    antarctica.tag_panels(axes, fontsize=STYLE.tag_size)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS11_additivity_{WINTER}')

**Figure S12. As Fig. 6, in summer (DJF).**

In [ ]:
#(t): ... in SUMMER
with plt.rc_context(STYLE.rc):
    panel_width = charts.panel_width_to_fit(len(zonal_columns), WIDTH)
    fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(zonal_columns),
                                                      n_legends=1,
                                                      panel_width=panel_width, panel_height=1.15, top_margin=0.6),
                     layout='constrained')
    axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(zonal_columns)).subplots(sharex=True, sharey='col',
                                                                                  squeeze=False)
    for ax in axes.ravel():
        ax.axhline(0, color='0.64', linewidth=0.8, zorder=1)
    for i, experiment in enumerate(FORCED_EXPERIMENTS):
        model_styles = charts.model_lines(experiment, zonal_models)
        for model in zonal_models:
            model_zonal_ds = zonal_ds.sel(model=model, season=SUMMER, experiment=experiment)
            model_color = model_styles[model]['color']
            charts.lines_grid(model_zonal_ds[zonal_columns].to_dataarray('column'), 'lat', None, 'column',
                              axes=axes[i], **model_styles[model])
            charts.plot_significant_dots(model_zonal_ds['mean_change'], model_zonal_ds['mean_pvalue'] < ALPHA, 'lat',
                                         ax=axes[i, 0], color=model_color)
            charts.plot_significant_dots(model_zonal_ds['width_change'], model_zonal_ds['width_pvalue'] < ALPHA, 'lat',
                                         ax=axes[i, 3], color=model_color)
            if ice_edges_da is not None:
                zp.ice_edge_lines_grid(ice_edges_da.sel(model=model, experiment=experiment, season=SUMMER).expand_dims(
                    column=zonal_columns), None, 'column', axes=axes[i], color=model_color)

    for ax, column in zip(axes[0], zonal_columns):
        ax.set_title(f'{rcp.EXTREME_TITLES[column]}\n{rc.EXTREMES[column]}', loc='left')
    for ax, experiment in zip(axes[:, 0], FORCED_EXPERIMENTS):
        ax.set_ylabel(f'{experiment}\n({VARIABLE.units})')
    for ax in axes[-1]:
        ax.set_xlabel('Latitude')
    axes[0, 0].xaxis.set_major_formatter(zp.latitude_label)
    axes[0, 0].set_xlim(float(zonal_ds['lat'].min()), float(zonal_ds['lat'].max()))
    extremes_ylim = (min(ax.get_ylim()[0] for ax in axes[0, 1:]), max(ax.get_ylim()[1] for ax in axes[0, 1:]))
    for ax in axes[0, 1:]:
        ax.set_ylim(extremes_ylim)
    #(c): One legend for every row: model k is the k-th shade of each row's colour, shown in historical's greys
    fig.legend(handles=[Line2D([], [], **style)
                        for style in charts.model_lines('historical', zonal_models).values()] + [
Line2D([], [], linestyle='none', marker='o', markersize=4, color='k', markeredgecolor='white',
    label=f'Significant (p < {ALPHA:g})'),
Line2D([], [], color='k', linewidth=1.0, linestyle=zp.EDGE_LINESTYLE, label='Sea-ice edge')],
loc='outside lower center', ncols=len(zonal_models) + 2, frameon=False)
    charts.tag_panels(axes)

In [ ]:
fr.save(fig, PAPER_DIR, f'figS12_zonal_{SUMMER}')

## What Was Saved

In [ ]:
pd.Series(sorted(path.name for path in PAPER_DIR.glob('*.pdf')), name='saved in ' + str(PAPER_DIR))